# Training a Deep Q-Network (DQN) to do CartPole Balancing

This notebook is part of [AI for Beginners Curriculum](http://aka.ms/ai-beginners). It is a companion to the [Policy Gradients and Actor-Critic notebook](CartPole-RL-PyTorch.ipynb), and it solves the same problem with a different family of algorithms - **value-based** RL. The implementation is inspired by the [official PyTorch tutorial on Reinforcement Learning (DQN)](https://pytorch.org/tutorials/intermediate/reinforcement_q_learning.html) and by the [original DQN paper](https://www.nature.com/articles/nature14236).

Instead of training a network that directly outputs the *policy* $\pi(a|s)$, we will train a network that estimates the **Q-function** $Q(s,a)$ - the total discounted reward we expect to receive if we take action $a$ in state $s$ and play well afterwards. The policy then becomes very simple: in each state take the action with the highest Q-value.

Our network will be trained using the **Bellman equation**:

$$Q(s,a) \approx r + \gamma \max_{a'} Q(s',a')$$

To make this training stable, we will use three ingredients that together make up the *Deep Q-Network* algorithm:

1. A **replay buffer** to train on random batches of past experience
1. A separate **target network** to compute stable targets
1. **$\epsilon$-greedy** exploration

> **Note**: This notebook does not open any simulation windows, so it can run anywhere (including Binder, Colab or a server). The training takes a few minutes on a CPU. At the end you can optionally watch the trained agent if you run the notebook locally.

We will start with the imports. The `gym` package is listed in `requirements.txt` of this course; if it is missing in your environment, install it with `pip install gym==0.26.2`.

In [ ]:
import copy
import random
from collections import deque, namedtuple

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import gym

# Fix random seeds to make the results reproducible
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## The CartPole Environment

Let's create the CartPole environment, as we did in the previous notebook. An environment has the following properties:

* **Action space** is the set of possible actions that we can perform at each step of the simulation. In CartPole there are 2 actions: push the cart left (0) or right (1).
* **Observation space** is the space of observations that we can make. In CartPole an observation is 4 numbers: position and velocity of the cart, angle and angular velocity of the pole.

> **Note**: Version 0.26 of Gym slightly changed the interface compared to older tutorials. `env.reset()` returns a tuple `(observation, info)`, and `env.step()` returns **five** values: `observation, reward, terminated, truncated, info`. The old `done` flag has been split in two:
> * `terminated` is `True` when the episode ended because the pole fell (or the cart left the track),
> * `truncated` is `True` when the episode was stopped by the time limit (500 steps in `CartPole-v1`) even though the pole was still balancing.

In [ ]:
env = gym.make("CartPole-v1")

state_size = env.observation_space.shape[0]
action_size = env.action_space.n

print(f"Action space: {env.action_space}")
print(f"Observation space: {env.observation_space}")

# Seed the environment (only needs to be done at the first reset)
state, info = env.reset(seed=SEED)
env.action_space.seed(SEED)
print(f"Initial state: {state}")

As a baseline, let's see what a **random** policy gets. In CartPole we receive a reward of 1 for each step the pole stays up, so the total reward is the length of the episode. The maximum is 500.

In [ ]:
def run_random_episode():
    env.reset()
    total_reward, done = 0, False
    while not done:
        _, reward, terminated, truncated, _ = env.step(env.action_space.sample())
        total_reward += reward
        done = terminated or truncated
    return total_reward

random_scores = [run_random_episode() for _ in range(50)]
print(f"Average reward of a random policy: {np.mean(random_scores):.1f}")

## The Q-Network

Our Q-network takes the **state** (4 numbers) as input and returns **one Q-value for each action** (2 numbers). Note that, unlike the policy network from the previous notebook, the output is not passed through softmax - Q-values are not probabilities, they are estimates of future reward and can be any number.

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_size, action_size, hidden_size=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, action_size),
        )

    def forward(self, state):
        return self.net(state)

# The network that we train
policy_net = QNetwork(state_size, action_size).to(device)
# The target network is a copy that is used to compute the targets (see below)
target_net = QNetwork(state_size, action_size).to(device)
target_net.load_state_dict(policy_net.state_dict())

## Replay Buffer

If we trained the network on steps in the order we collect them, consecutive examples would be almost identical (the cart moves only a little between two steps), and the network would quickly "forget" the earlier parts of the game. To avoid that, we store each step of the experiment as a **transition** $(s, a, r, s', \text{terminated})$ in a memory called **replay buffer**, and then train on small **random batches** sampled from it. When the buffer is full, the oldest transitions are dropped.

> ✅ Think about it: how is this similar to shuffling the dataset when we train a normal classifier? What is different?

In [ ]:
Transition = namedtuple("Transition", ["state", "action", "reward", "next_state", "terminated"])

class ReplayBuffer:
    def __init__(self, capacity):
        self.memory = deque(maxlen=capacity)

    def push(self, *args):
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        # Take a random batch and convert it into tensors
        batch = Transition(*zip(*random.sample(self.memory, batch_size)))
        states = torch.tensor(np.array(batch.state), dtype=torch.float32, device=device)
        actions = torch.tensor(batch.action, dtype=torch.int64, device=device).unsqueeze(1)
        rewards = torch.tensor(batch.reward, dtype=torch.float32, device=device)
        next_states = torch.tensor(np.array(batch.next_state), dtype=torch.float32, device=device)
        terminated = torch.tensor(batch.terminated, dtype=torch.float32, device=device)
        return states, actions, rewards, next_states, terminated

    def __len__(self):
        return len(self.memory)

## Choosing Actions: $\epsilon$-greedy

During training we need to balance **exploration** and **exploitation**. With probability $\epsilon$ we take a random action, and otherwise we take the action with the highest Q-value (the *greedy* action). We start with $\epsilon=1$ (pure exploration, because the network knows nothing yet) and decrease it exponentially towards a small value, so that the agent relies more and more on what it has learned.

In [ ]:
EPS_START, EPS_END, EPS_DECAY = 1.0, 0.05, 3000   # EPS_DECAY is measured in environment steps

def epsilon_by_step(step):
    return EPS_END + (EPS_START - EPS_END) * np.exp(-step / EPS_DECAY)

def select_action(state, epsilon):
    if random.random() < epsilon:
        return env.action_space.sample()               # explore
    with torch.no_grad():
        state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        return policy_net(state_t).argmax(dim=1).item()  # exploit

for step in [0, 1000, 3000, 6000, 12000]:
    print(f"step {step:>5}: epsilon = {epsilon_by_step(step):.2f}")

## One Training Step

Here is the heart of the algorithm. We take a random batch from the replay buffer and:

1. Compute the Q-values that our network currently predicts for the actions that were actually taken: $Q(s,a)$.
1. Compute the **Bellman target** $y = r + \gamma \max_{a'} Q_{target}(s',a')$ using the **target network**. If the episode really ended in $s'$ (the pole fell), there is no future reward, so $y = r$. The target is treated as a constant - we do not propagate gradients through it.
1. Minimize the difference between $Q(s,a)$ and $y$. We use the *Huber loss* (`smooth_l1_loss`), which is less sensitive to occasional large errors than the squared error.

Why do we need a separate target network? If we used `policy_net` to compute the targets, then every update would also shift the target we are chasing - like trying to hit a moving target. The target network is only synchronized with `policy_net` once in a while, so the targets stay stable in between.

> **Note**: We use the `terminated` flag (not `truncated`) to cut off the future reward. When an episode is only truncated by the 500-step time limit, the pole is still balancing, so the state still has a future value, and we should keep bootstrapping from it.

In [ ]:
GAMMA = 0.99          # discount factor
BATCH_SIZE = 64
LEARNING_RATE = 5e-4

optimizer = torch.optim.Adam(policy_net.parameters(), lr=LEARNING_RATE)

def optimize_model(memory):
    states, actions, rewards, next_states, terminated = memory.sample(BATCH_SIZE)

    # Q(s,a) predicted by the policy network for the actions that were taken
    q_values = policy_net(states).gather(1, actions).squeeze(1)

    # Bellman target: r + gamma * max_a' Q_target(s',a'), or just r if the episode has ended
    with torch.no_grad():
        next_q_values = target_net(next_states).max(dim=1).values
        targets = rewards + GAMMA * next_q_values * (1 - terminated)

    loss = nn.functional.smooth_l1_loss(q_values, targets)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
    optimizer.step()
    return loss.item()

## The Training Loop

Now we put everything together. In each episode we repeat the following until the episode ends:

1. Choose an action with $\epsilon$-greedy policy and perform it in the environment.
1. Store the resulting transition in the replay buffer.
1. Once the buffer contains enough examples, do one training step on a random batch.
1. Every `TARGET_UPDATE` steps, copy the weights of `policy_net` into `target_net`.

We will train for 400 episodes, which takes a few minutes on a CPU. Notice how the reward grows as $\epsilon$ goes down and the Q-network gets better. The curve is not perfectly smooth - this is typical for DQN, and the agent may even get worse for a while after reaching a good result. For this reason, we also remember the weights of the network at the moment when the average reward of the last 20 episodes was the highest.

In [ ]:
NUM_EPISODES = 400
BUFFER_SIZE = 10000
WARMUP_STEPS = 500     # fill the buffer with some experience before we start learning
TARGET_UPDATE = 500    # copy policy_net -> target_net every 500 steps

memory = ReplayBuffer(BUFFER_SIZE)
episode_rewards = []
total_steps = 0
best_avg, best_weights = 0, copy.deepcopy(policy_net.state_dict())

for episode in range(NUM_EPISODES):
    state, _ = env.reset()
    episode_reward, done = 0, False

    while not done:
        epsilon = epsilon_by_step(total_steps)
        action = select_action(state, epsilon)
        next_state, reward, terminated, truncated, _ = env.step(action)
        memory.push(state, action, reward, next_state, float(terminated))

        state = next_state
        episode_reward += reward
        total_steps += 1
        done = terminated or truncated

        if len(memory) >= WARMUP_STEPS:
            optimize_model(memory)

        if total_steps % TARGET_UPDATE == 0:
            target_net.load_state_dict(policy_net.state_dict())

    episode_rewards.append(episode_reward)

    # DQN can sometimes "forget" what it has learned, so we remember the best weights seen so far
    if len(episode_rewards) >= 20 and np.mean(episode_rewards[-20:]) > best_avg:
        best_avg = np.mean(episode_rewards[-20:])
        best_weights = copy.deepcopy(policy_net.state_dict())

    if episode % 20 == 0 or episode == NUM_EPISODES - 1:
        avg = np.mean(episode_rewards[-20:])
        print(f"Episode {episode:>3}: reward = {episode_reward:>5.0f}, average of last 20 = {avg:>6.1f}, epsilon = {epsilon:.2f}")

## Training Curve

Let's plot the reward obtained in each episode, together with a moving average that makes the trend easier to see. The reward during training is lower than the real quality of the agent, because during training the agent still takes random actions with probability $\epsilon$.

In [ ]:
window = 20
moving_avg = np.convolve(episode_rewards, np.ones(window) / window, mode="valid")

plt.figure(figsize=(10, 4))
plt.plot(episode_rewards, alpha=0.4, label="Reward per episode")
plt.plot(np.arange(window - 1, len(episode_rewards)), moving_avg, linewidth=2, label=f"Moving average ({window} episodes)")
plt.axhline(np.mean(random_scores), color="gray", linestyle="--", label="Random policy")
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.title("DQN on CartPole-v1")
plt.legend()
plt.show()

## Evaluation

To see how good the trained agent really is, we first load the best weights that we saved during training, and then run the agent with $\epsilon=0$, i.e. always taking the greedy action. We play 10 episodes and report the results. A perfect score in `CartPole-v1` is 500.

If you run the notebook on your computer, you can also set `render=True` to watch the agent in a window.

In [ ]:
def evaluate(n_episodes=10, render=False):
    eval_env = gym.make("CartPole-v1", render_mode="human" if render else None)
    scores = []
    for _ in range(n_episodes):
        state, _ = eval_env.reset()
        total_reward, done = 0, False
        while not done:
            action = select_action(state, epsilon=0.0)
            state, reward, terminated, truncated, _ = eval_env.step(action)
            total_reward += reward
            done = terminated or truncated
        scores.append(total_reward)
    eval_env.close()
    return scores

policy_net.load_state_dict(best_weights)
print(f"Best average reward of 20 training episodes: {best_avg:.1f}")

scores = evaluate(n_episodes=10)
print(f"Evaluation rewards: {scores}")
print(f"Average reward: {np.mean(scores):.1f} (random policy: {np.mean(random_scores):.1f})")

In [ ]:
env.close()

## Takeaway

In this notebook we have trained an agent to balance the pole using a completely different approach than in the previous notebook: instead of learning the policy directly, we have learned the **value of each action** and derived the policy from it. The three key ideas of Deep Q-Networks were:

* **Replay buffer** - learn from random batches of past experience instead of consecutive steps,
* **Target network** - compute the Bellman targets with a slowly-changing copy of the network,
* **$\epsilon$-greedy exploration** - gradually move from exploration to exploitation.

DQN is known to be sensitive to hyper-parameters, so the results vary from run to run - you may notice that the reward sometimes drops after reaching a high value. Real-world implementations use many additional improvements, such as *Double DQN*, *prioritized replay* or *dueling networks*.

## ✅ Challenge

Experiment with the algorithm:

* Change `SEED` and run the training several times. How much do the results differ?
* What happens if you set `TARGET_UPDATE = 1` (so the target network is always the same as the policy network)? Or if you make the replay buffer very small (e.g. 500)? Which of the DQN ingredients seems to matter the most?
* Try different values of `GAMMA`, `LEARNING_RATE` and `EPS_DECAY`.
* Because Gym environments share the same interface, you can try to train the same code on another environment with a discrete action space - just change the name in `gym.make`, and make sure that the sizes of the input and output of the network match the new environment.